# Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- **Start Simple**: Begin with a simple architecture and gradually increase complexity if needed.
- **Grid Search/Random Search**: Use `grid search` or `random search` to try different architectures.
- **Cross-Validation**: Use cross-validation to evaluate the performance of different architectures.
- **Heuristics and Rules of Thumb**: Some heuristics and empirical rules can provide starting points, such as:-
  -  The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [1]:
import pandas as pd
import pickle
from itertools import product

from sklearn.model_selection import train_test_split as TTS, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
# from tensorflow.keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier, KerasRegressor

I0000 00:00:1790052346.076740 1276468 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790052346.077063 1276468 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790052346.104213 1276468 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790052347.076812 1276468 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

In [2]:
def generate_hidden_layers(neurons, layers):

    architectures = []

    for n_layers in layers:
        layer_combinations = product(neurons, repeat=n_layers)
        architectures.extend(layer_combinations)

    return architectures

In [3]:
def  create_model(hidden_layers=(16,1), type='classifier'):
    model = Sequential()

    # Input layer
    model.add(Input(shape=(X_train.shape[1],)))

    # Find out hidden layers
    for units in hidden_layers:
        model.add(Dense(units=units, activation='relu'))

    # Output layer
    if type == 'classifier':
        model.add(Dense(units=1, activation='sigmoid'))

        # Configure neural network for training
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        
    elif type == 'regressor':
        model.add(Dense(units=1))

        # Configure neural network for training
        model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

    return model

In [8]:
def hyperparameters_tuning(X, y, param_grid, type="classifier"):

    if type == "classifier":
        estimator = KerasClassifier(model=create_model, type=type, verbose=0)

    elif type == "regressor":
        estimator = KerasRegressor(model=create_model, type=type, verbose=0)

    else:
        raise ValueError("Type must be 'classifier' or 'regressor'.")

    grid = GridSearchCV(estimator=estimator,param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

    grid_result = grid.fit(X, y)

    return grid_result

### Hyperparameter Tuning for Classification

In [13]:
data=pd.read_csv('./data/churn_modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = TTS(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('./preprocessors/classification/gender_label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('./preprocessors/classification/onehot_geo_encoder.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('./preprocessors/classification/standard_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [ ]:
neurons = [16, 32, 64, 128]
layers = [1, 2, 3, 4]

architectures = generate_hidden_layers(neurons, layers)

param_grid = {"model__hidden_layers": architectures,
              "epochs": [10, 50, 100]}

grid_result = hyperparameters_tuning(X=X_train, y=y_train, param_grid=param_grid, type="classifier")

print("Best Score:", grid_result.best_score_)
print("Best Parameters:", grid_result.best_params_)

In [24]:
model = Sequential([Dense(16, activation='relu', input_shape=(X_train.shape[1],)), # HL connected with Input Layer
                    Dense(1, activation='sigmoid')])  # Output Layer
model.save('./dl_model/classification_model.h5')
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 16)             │           208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225 (900.00 B)

 Trainable params: 225 (900.00 B)

 Non-trainable params: 0 (0.00 B)

### Hyperparameter Tuning for Regression

In [5]:
data=pd.read_csv('./data/churn_modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']

X_train, X_test, y_train, y_test = TTS(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('./preprocessors/classification/gender_label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('./preprocessors/classification/onehot_geo_encoder.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('./preprocessors/classification/standard_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [9]:
neurons = [16, 32, 64, 128]
layers = [1, 2, 3, 4]

architectures = generate_hidden_layers(neurons, layers)

param_grid = {"model__hidden_layers": architectures,
              "epochs": [10, 50, 100]}

grid_result = hyperparameters_tuning(X=X_train, y=y_train, param_grid=param_grid, type="regressor")

print("Best Score:", grid_result.best_score_)
print("Best Parameters:", grid_result.best_params_)

Fitting 3 folds for each of 1020 candidates, totalling 3060 fits


I0000 00:00:1790052485.286824 1277724 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790052485.288031 1277724 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790052485.322438 1277732 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790052485.323360 1277732 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790052485.364286 1277724 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the follow

[CV] END ..............epochs=10, model__hidden_layers=(32,); total time=   7.0s
[CV] END ..............epochs=10, model__hidden_layers=(16,); total time=   7.7s
[CV] END .............epochs=10, model__hidden_layers=(128,); total time=   7.7s
[CV] END ..............epochs=10, model__hidden_layers=(16,); total time=   8.3s
[CV] END ..............epochs=10, model__hidden_layers=(16,); total time=   7.5s
[CV] END ..............epochs=10, model__hidden_layers=(64,); total time=   8.5s
[CV] END ..............epochs=10, model__hidden_layers=(64,); total time=   8.4s
[CV] END ..............epochs=10, model__hidden_layers=(32,); total time=   8.7s
[CV] END ...........epochs=10, model__hidden_layers=(16, 16); total time=   8.0s
[CV] END ...........epochs=10, model__hidden_layers=(16, 16); total time=   7.7s
[CV] END ..............epochs=10, model__hidden_layers=(32,); total time=   9.0s
[CV] END ...........epochs=10, model__hidden_layers=(16, 32); total time=   8.1s
[CV] END .............epochs

/home/htet-aung-lynn/Study/E2E-Churn-Prediction-with-ANN/venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END ..epochs=10, model__hidden_layers=(16, 128, 64, 32); total time=  10.0s
[CV] END ..epochs=10, model__hidden_layers=(16, 128, 64, 32); total time=   9.9s
[CV] END ..epochs=10, model__hidden_layers=(16, 128, 64, 64); total time=   8.4s
[CV] END ..epochs=10, model__hidden_layers=(16, 128, 64, 64); total time=   8.3s
[CV] END ..epochs=10, model__hidden_layers=(16, 128, 64, 64); total time=   7.7s
[CV] END .epochs=10, model__hidden_layers=(16, 128, 64, 128); total time=   7.4s
[CV] END .epochs=10, model__hidden_layers=(16, 128, 64, 128); total time=   7.3s


...

[CV] END .epochs=100, model__hidden_layers=(128, 64, 32, 32); total time= 1.2min
[CV] END .epochs=100, model__hidden_layers=(128, 64, 32, 16); total time= 1.3min
[CV] END .epochs=100, model__hidden_layers=(128, 64, 32, 32); total time= 1.3min
[CV] END .epochs=100, model__hidden_layers=(128, 64, 32, 32); total time= 1.3min
[CV] END .epochs=100, model__hidden_layers=(128, 64, 32, 64); total time= 1.3min
[CV] END .epochs=100, model__hidden_layers=(128, 64, 32, 64); total time= 1.3min
[CV] END .epochs=100, model__hidden_layers=(128, 64, 32, 64); total time= 1.3min
[CV] END epochs=100, model__hidden_layers=(128, 64, 32, 128); total time= 1.3min
[CV] END epochs=100, model__hidden_layers=(128, 64, 32, 128); total time= 1.3min
[CV] END epochs=100, model__hidden_layers=(128, 64, 32, 128); total time= 1.3min
[CV] END .epochs=100, model__hidden_layers=(128, 64, 64, 16); total time= 1.3min
[CV] END .epochs=100, model__hidden_layers=(128, 64, 64, 16); total time= 1.3min
[CV] END .epochs=100, model_

I0000 00:00:1790059996.563966 3030871 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790059996.565178 3030871 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790059996.628297 3030871 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790059999.021286 3030871 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 16); total time= 1.4min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 16); total time= 1.5min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 32); total time= 1.4min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 16); total time= 1.5min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 32); total time= 1.4min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 32); total time= 1.4min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 64); total time= 1.4min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 64); total time= 1.5min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 64); total time= 1.5min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 128); total time= 1.5min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 128); total time= 1.5min
[CV] END epochs=100, model__hidden_layers=(128, 128, 32, 128); total time= 1.5min
[CV] END epochs=100, mode

E0000 00:00:1790060150.615460 1276468 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Best Score: -0.013663125191441639
Best Parameters: {'epochs': 100, 'model__hidden_layers': (64, 64)}


In [ ]:
# {'epochs': 100, 'model__hidden_layers': (64, 64)}
print("Best Score:", grid_result.best_score_)
print("Best Parameters:", grid_result.best_params_)

In [13]:
best_model = grid_result.best_estimator_.model_
best_model.save('regression_model.h5')
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,173 (59.27 KB)

 Trainable params: 5,057 (19.75 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 10,116 (39.52 KB)